# IS 362 – Project 3
**Bria King**

## Introduction
In this project, I used the SQLite version of the Chinook database to retrieve customer purchase information. I joined five tables: Customer, Invoice, InvoiceLine, Track, and Album. The results were loaded into a pandas DataFrame showing each customer's last name, first name, purchased track names, and album titles, sorted by last name and then first name.

## Database Setup
The file `Chinook_Sqlite.sqlite` must be in the same folder as this notebook. This project uses pandas and Python's built-in sqlite3 library. No additional database driver is needed. The database connection uses a file URI with `mode=ro` to open the database in read-only mode.

Database source: https://github.com/lerocha/chinook-database

## Step 1: Connect to the Database
I used Python's sqlite3 library to connect to the Chinook database in read-only mode. I displayed the table names to confirm that the database loaded.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

# Locate the database in the same folder as this notebook.
db_path = Path("Chinook_Sqlite.sqlite")

# Check that the file exists before connecting.
if not db_path.is_file():
    raise FileNotFoundError("Place Chinook_Sqlite.sqlite in the notebook folder.")

# Connect to the existing database.
conn = sqlite3.connect(db_path.resolve().as_uri() + "?mode=ro", uri=True)

# Display the database's tables.
pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;",
    conn
)

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


## Step 2: Retrieve Customer Purchases
I joined Customer, Invoice, InvoiceLine, Track, and Album using their matching ID fields. The query retrieves customer names, purchased track names, and album titles, sorted by customer last name and then first name. I loaded the results into a pandas DataFrame and displayed the first five rows.

In [2]:
# Join five tables to retrieve each customer's purchased tracks and albums.
query = """
SELECT
    c.LastName,
    c.FirstName,
    t.Name AS TrackName,
    a.Title AS AlbumTitle
FROM Customer AS c
INNER JOIN Invoice AS i
    ON c.CustomerId = i.CustomerId
INNER JOIN InvoiceLine AS il
    ON i.InvoiceId = il.InvoiceId
INNER JOIN Track AS t
    ON il.TrackId = t.TrackId
INNER JOIN Album AS a
    ON t.AlbumId = a.AlbumId
ORDER BY
    c.LastName,
    c.FirstName;
"""

# Load the query results into a pandas DataFrame.
purchases_df = pd.read_sql_query(query, conn)

# Display the first five rows.
purchases_df.head()

,LastName,FirstName,TrackName,AlbumTitle
0,Almeida,Roberto,Right Next Door to Hell,Use Your Illusion I
1,Almeida,Roberto,In The Evening,In Through The Out Door
2,Almeida,Roberto,Fool In The Rain,In Through The Out Door
3,Almeida,Roberto,Saudade Dos Aviões Da Panair (Conversando No Bar),Minas
4,Almeida,Roberto,Caso Você Queira Saber,Minas


## Step 3: Check the Results
I checked the DataFrame's dimensions and confirmed that the customer names are sorted correctly. The results show 2,240 rows and four columns, and the sorting check returned True.

In [3]:
# Check the number of purchase records and columns.
print("DataFrame dimensions:", purchases_df.shape)

# Confirm customers are sorted by last name, then first name.
is_sorted = purchases_df[["LastName", "FirstName"]].equals(
    purchases_df[["LastName", "FirstName"]]
    .sort_values(["LastName", "FirstName"], kind="stable")
    .reset_index(drop=True)
)

print("Customers sorted correctly:", is_sorted)

DataFrame dimensions: (2240, 4)
Customers sorted correctly: True


## Conclusion
I joined five tables from the Chinook database and loaded the results into a pandas DataFrame. The DataFrame contains 2,240 purchase records and four columns: LastName, FirstName, TrackName, and AlbumTitle. The results are sorted by customer last name and then first name, and the sorting check returned True. The first five rows are displayed as a preview.

In [4]:
# Close the database connection after retrieving the results.
conn.close()